In [3]:
import openpyxl
import numpy as np
import pandas as pd
from pathlib import Path


In [4]:
root_folder = Path("./data")
output_folder = Path("./data_anonymized")

students = pd.read_excel(root_folder / "students.xlsx")

students.rename(columns={
    'Endereço de e-mail': 'email',
    'EMAIL (@estudante.se.df.gov.br)': 'email'
}, inplace=True)

for file_path in root_folder.rglob("*.xlsx"):
    if file_path.name == "students.xlsx":
        continue

    df = pd.read_excel(file_path)

    df.rename(columns={
        'Endereço de e-mail': 'email',
        'EMAIL (@estudante.se.df.gov.br)': 'email'
    }, inplace=True)

    df_anonymize = (
        df.merge(alunos[['email', 'id']], on='email', how='left')
          .drop(columns=['email'])
    )

    relative_path = file_path.relative_to(root_folder)

    new_name = relative_path.stem + "_anonymized.xlsx"

    output_path = output_folder / relative_path.parent
    output_path.mkdir(parents=True, exist_ok=True)

    df_anonymize.to_excel(output_path / new_name, index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'data/students.xlsx'

In [80]:
df = pd.read_csv("./data/provas.csv")
df.head()

answers = df.loc[:, "0":"39"]
answers_numeric = answers.apply(lambda col: col.astype("category").cat.codes)
answers_numeric.head()

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42)

clusters = kmeans.fit_predict(answers_numeric)

df["cluster"] = clusters

import numpy as np

def avg_distance(group):
    X = group.values
    dists = []

    for i in range(len(X)):
        for j in range(i+1, len(X)):
            d = np.sum(X[i] != X[j])
            dists.append(d)

    return np.mean(dists)

cluster0 = answers_numeric[df.cluster == 0]
cluster1 = answers_numeric[df.cluster == 1]

print("cluster 0 (não colaram) média:", avg_distance(cluster0))
print("cluster 1 (colaram) média:", avg_distance(cluster1))

print(df["cluster"].value_counts(ascending=True))

cluster 0 (não colaram) média: 5.469924812030075
cluster 1 (colaram) média: 5.100156494522691
cluster
0    57
1    72
Name: count, dtype: int64
